## Act 1:A NUMBER THAT CHOOSES THE STORY


Investigate whether the NYC Taxi ecosystem is functioning normally or drifting into inefficiency,
despite stable revenue and trip counts.

Key Objective:
Identify behavioral shifts in drivers and passengers.

Approach:
- Use mathematical constraint to define analysis window
- Compare before, during, after periods
- Detect hidden inefficiencies using derived metrics

In [1]:
%pip install pandas pyarrow matplotlib

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-win_amd64.whl.metadata (3.0 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp313-cp313-win_amd64.whl.metadata (119 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl (9.7 MB)
Using cached pyarrow-24.0.0-cp313-cp313-win_amd64.whl (27.3 MB)
Using c


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import sys
!"{sys.executable}" -m pip install matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# lets assume the digits to be 2,5,7,8
W, X, Y, Z = 2, 5, 7, 8 

# Step 1: Sum
total = W + X + Y + Z

# Step 2: Digital Root
def digital_root(n):
    while n > 9:
        n = sum(map(int, str(n)))
    return n

R = digital_root(total)

# Critical points
x1 = R - 1
x2 = R + 1

print(f"R = {R}")
print(f"Analysis Window: {x1} to {x2}")

R = 4
Analysis Window: 3 to 5


In [3]:
import pandas as pd

files = [
    "yellow_tripdata_2023-02.parquet",
    "yellow_tripdata_2023-03.parquet",
    "yellow_tripdata_2023-04.parquet",
    "yellow_tripdata_2023-05.parquet",
    "yellow_tripdata_2023-06.parquet"
]

df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print("Shape:", df.shape)
df.head()

Shape: (16426854, 19)


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,1,2023-02-01 00:32:53,2023-02-01 00:34:34,2.0,0.30,1.0,N,142,163,2,4.4,3.50,0.5,0.0,0.0,1.0,9.40,2.5,0.00
1,2,2023-02-01 00:35:16,2023-02-01 00:35:30,1.0,0.00,1.0,N,71,71,4,-3.0,-1.00,-0.5,0.0,0.0,-1.0,-5.50,0.0,0.00
2,2,2023-02-01 00:35:16,2023-02-01 00:35:30,1.0,0.00,1.0,N,71,71,4,3.0,1.00,0.5,0.0,0.0,1.0,5.50,0.0,0.00
3,1,2023-02-01 00:29:33,2023-02-01 01:01:38,0.0,18.80,1.0,N,132,26,1,70.9,2.25,0.5,0.0,0.0,1.0,74.65,0.0,1.25
4,2,2023-02-01 00:12:28,2023-02-01 00:25:46,1.0,3.22,1.0,N,161,145,1,17.0,1.00,0.5,3.3,0.0,1.0,25.30,2.5,0.00


In [ ]:
#cleaning the data 
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

df = df[df['trip_distance'] > 0]
df = df[df['fare_amount'] > 0]

df = df[df['trip_distance'] < 100]
df = df[df['fare_amount'] < 500]

In [5]:
# FEATURE ENGINEERING:  Trip duration
df['trip_duration'] = (
    (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime'])
    .dt.total_seconds() / 60
)

# Revenue efficiency
df['revenue_per_mile'] = df['fare_amount'] / df['trip_distance']

# Tip behavior
df['tip_percentage'] = (df['tip_amount'] / df['fare_amount']) * 100

# Time features
df['month'] = df['tpep_pickup_datetime'].dt.month
df['hour'] = df['tpep_pickup_datetime'].dt.hour

In [ ]:
#applying mathamatical code R = 4 - window = 3 to 5
df_before = df[df['month'] < 3]
df_window = df[(df['month'] >= 3) & (df['month'] <= 5)]
df_after  = df[df['month'] > 5]

In [ ]:
#filtering based on mathamatical window 
df_window = df[(df['month'] >= x1) & (df['month'] <= x2)]

df_before = df[df['month'] < x1]
df_after  = df[df['month'] > x2]

print("Window shape:", df_window.shape)

Window shape: (9986675, 24)


In [ ]:
#Comparing Behavior 
def summarize(data, name):
    print(f"\n--- {name} ---")
    print("Trips:", len(data))
    print("Avg Distance:", data['trip_distance'].mean())
    print("Avg Fare:", data['fare_amount'].mean())
    print("Avg Duration:", data['trip_duration'].mean())
    print("Revenue/Mile:", data['revenue_per_mile'].mean())
    print("Tip %:", data['tip_percentage'].mean())

summarize(df_before, "BEFORE")
summarize(df_window, "STRESS ZONE")
summarize(df_after, "AFTER")


--- BEFORE ---
Trips: 2850513
Avg Distance: 3.355073311365358
Avg Fare: 18.440992382774606
Avg Duration: 16.118553788972957
Revenue/Mile: 10.265695494123614
Tip %: 20.734438038251653

--- STRESS ZONE ---
Trips: 9986675
Avg Distance: 3.53936860066038
Avg Fare: 19.634087890113577
Avg Duration: 17.519243401832945
Revenue/Mile: 10.742874843202321
Tip %: 21.03598932027817

--- AFTER ---
Trips: 3231063
Avg Distance: 3.6272356589766277
Avg Fare: 20.129989319304514
Avg Duration: 17.836688225103213
Revenue/Mile: 11.09100301087274
Tip %: 21.294864264364254
